# Quantum Harmonic Oscillator — PINN Solution

This notebook demonstrates a Physics-Informed Neural Network solution to the **time-independent Schrödinger equation** for the quantum harmonic oscillator:

$$
\hat{H}\psi = E\psi, \quad \hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + \frac{1}{2}m\omega^2 x^2
$$

With $\hbar = m = \omega = 1$, the analytic ground-state energy is $E_0 = \frac{1}{2}$.

The PINN is trained by minimizing the PDE residual:

$$
\mathcal{L}_{\text{PDE}} = \left\|\hat{H}\psi_{\theta} - E_{\theta}\psi_{\theta}\right\|_2^2
$$

subject to boundary conditions $\psi(\pm L) = 0$ and normalization $\int|\psi|^2 dx = 1$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.pinn import PINN
from src.physics import harmonic_potential, tise_residual_1d, PINNLoss
from src.data import sample_interior, harmonic_oscillator_ground_state

plt.style.use('dark_background')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
X_MIN, X_MAX = -5.0, 5.0
N_COLLOCATION = 1000
N_EPOCHS = 8000
LR = 1e-3
HIDDEN_DIM = 64
N_LAYERS = 4

# ── Build model + energy parameter ───────────────────────────────────────
model = PINN(in_dim=1, out_dim=1, hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS).to(device)
energy = torch.tensor([1.5], requires_grad=True, device=device)

optimizer = torch.optim.Adam(list(model.parameters()) + [energy], lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=500, factor=0.5)
loss_fn = PINNLoss(lambda_pde=1.0, lambda_bc=10.0)

print(f'Model parameters: {model.count_parameters():,}')
print(f'Initial energy guess: {energy.item():.4f}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────
history = []

bc_x = torch.FloatTensor([[X_MIN], [X_MAX]]).to(device).requires_grad_(True)
bc_t = torch.zeros_like(bc_x).requires_grad_(True)

for epoch in range(1, N_EPOCHS + 1):
    optimizer.zero_grad()

    x_col, t_col = sample_interior(N_COLLOCATION, (X_MIN, X_MAX), (0.0, 0.0), device)

    psi_col = model(x_col, torch.zeros_like(x_col))
    pde_res = tise_residual_1d(psi_col, x_col, energy, harmonic_potential)

    psi_bc = model(bc_x, bc_t)

    losses = loss_fn([pde_res], [psi_bc], [], [])
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step(losses['total'].item())

    if epoch % 1000 == 0:
        print(f'Epoch {epoch:5d} | total={losses["total"].item():.4e} '
              f'| pde={losses["pde"].item():.4e} '
              f'| E={energy.item():.4f}')
        history.append({'epoch': epoch, 'total': losses['total'].item(), 'energy': energy.item()})

In [ ]:
# ── Evaluation and visualization ───────────────────────────────────────────
model.eval()
x_test = np.linspace(X_MIN, X_MAX, 500)
psi_analytic = harmonic_oscillator_ground_state(x_test, omega=1.0)

x_t = torch.FloatTensor(x_test).unsqueeze(1).to(device)
with torch.no_grad():
    psi_pred = model(x_t, torch.zeros_like(x_t)).cpu().numpy().flatten()

# Normalize sign: match sign of analytic
if np.sign(psi_pred[len(psi_pred)//2]) != np.sign(psi_analytic[len(psi_analytic)//2]):
    psi_pred = -psi_pred

# Relative L2 error
l2_error = np.linalg.norm(psi_pred - psi_analytic) / np.linalg.norm(psi_analytic)
print(f'Predicted E₀ = {energy.item():.5f}  (analytic = 0.50000)')
print(f'Relative L² error = {l2_error:.5f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

# Wavefunction comparison
axes[0].plot(x_test, psi_analytic, '--', color='#58a6ff', linewidth=2, label='Analytic ψ₀')
axes[0].plot(x_test, psi_pred,      '-',  color='#3fb950', linewidth=2, label='PINN prediction')
axes[0].fill_between(x_test, psi_analytic, psi_pred, alpha=0.15, color='#ff7b72')
axes[0].set_xlabel('x', color='#8b949e')
axes[0].set_ylabel('ψ(x)', color='#8b949e')
axes[0].set_title(f'Ground State Wavefunction  (L² error={l2_error:.4f})', color='#e6edf3')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

# Training loss
epochs_hist = [h['epoch'] for h in history]
total_hist  = [h['total'] for h in history]
axes[1].semilogy(epochs_hist, total_hist, color='#58a6ff', linewidth=2)
axes[1].set_xlabel('Epoch', color='#8b949e')
axes[1].set_ylabel('Total Loss', color='#8b949e')
axes[1].set_title('Training Convergence', color='#e6edf3')

plt.tight_layout()
plt.savefig('../outputs/harmonic_oscillator_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → outputs/harmonic_oscillator_results.png')